<a href="https://colab.research.google.com/github/adrinorosario/legal-pragmatic-inference/blob/main/hybrid_clause_and_term_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hybrid Clause and Vague Term Extraction

Working on this approach here, the goal is to extract the contractual clauses and the underlying vague terms in the particular clauses. For this, we will be used 3 datasets:

*   [CUAD](https://huggingface.co/datasets/theatticusproject/cuad)
*   [ContractNLI](https://huggingface.co/datasets/kiddothe2b/contract-nli); extensive information can be found [here](https://stanfordnlp.github.io/contract-nli/). Refer to the paper [here](https://arxiv.org/pdf/2110.01799)
*   [LEDGAR](https://huggingface.co/datasets/coastalcph/lex_glue) (the subset in LexGLUE)

Extracting the anchor clauses and the vague terms will form the first two parts of the [triplet dataset](https://github.com/adrinorosario/legal-pragmatic-inference/blob/main/docs/research/dataset_construction.md).




In [ ]:
%%capture
!pip install

## Extraction from CUAD

For this, we are using the JSON file from the HuggingFace page of the CUAD Dataset which can be found [here](https://huggingface.co/datasets/theatticusproject/cuad/tree/main/CUAD_v1)

In [2]:
# read the json file

import json
import requests

# fetch the raw CUAD JSON directly from the huggingface page (https://huggingface.co/datasets/theatticusproject/cuad/blob/main/CUAD_v1/)
cuad_v1_json = "/content/CUAD_v1.json"

with open(cuad_v1_json, "r") as file:
  data = json.load(file)

data.keys()

dict_keys(['version', 'data'])

In [3]:
len(data["data"]) # contains 510 entries

510

In [4]:
cuad_data = data["data"]
cuad_data[0].keys() # each entry, i.e., a contract contains the title and the paragraphs in it

dict_keys(['title', 'paragraphs'])

In [5]:
print(f"Type of cuad_data[0]['paragraphs']: {type(cuad_data[0]["paragraphs"])}")
print(f"Length of cuad_data[0]['paragraphs]: {len(cuad_data[0]["paragraphs"])}")

print(f"\nType of cuad_data[0]['paragraphs'][0]: {type(cuad_data[0]['paragraphs'][0])}")
print(f"Length of cuad_data[0]['paragraphs'][0]: {len(cuad_data[0]['paragraphs'][0])}")
print(f"Keys of cuad_data[0]['paragraphs'][0]: {cuad_data[0]["paragraphs"][0].keys()}")

Type of cuad_data[0]['paragraphs']: <class 'list'>
Length of cuad_data[0]['paragraphs]: 1

Type of cuad_data[0]['paragraphs'][0]: <class 'dict'>
Length of cuad_data[0]['paragraphs'][0]: 2
Keys of cuad_data[0]['paragraphs'][0]: dict_keys(['qas', 'context'])


1.   You are accessing each contract's paragraphs, which is a **list**.
2.   Each list of paragraphs contains a dictionary, which has two keys: **qas** and **context**



In [6]:
print(f"Type of cuad_data[0]['paragraphs'][0]['qas']: {type(cuad_data[0]['paragraphs'][0]['qas'])}")
print(f"Length of cuad_data[0]['paragraphs'][0]['qas']: {len((cuad_data[0]['paragraphs'][0]['qas']))}")

print("\nLooking at a single qas:")
print(cuad_data[0]['paragraphs'][0]['qas'][0])
print(f"Keys in a single qas: {cuad_data[0]['paragraphs'][0]['qas'][0].keys()}")

Type of cuad_data[0]['paragraphs'][0]['qas']: <class 'list'>
Length of cuad_data[0]['paragraphs'][0]['qas']: 41

Looking at a single qas:
{'answers': [{'text': 'DISTRIBUTOR AGREEMENT', 'answer_start': 44}], 'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Document Name', 'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract', 'is_impossible': False}
Keys in a single qas: dict_keys(['answers', 'id', 'question', 'is_impossible'])


In [38]:
cuad_data[0]['paragraphs'][0]['qas'][4]["answers"][0]

{'text': 'The term of this  Agreement  shall be ten (10)                            years (the "Term")  which shall  commence on the date                            upon which the Company  delivers to  Distributor  the                            last Sample, as defined  hereinafter.',
 'answer_start': 5268}

1.   **qas** is a list. Each item in the list is a dictionary.
2.   A single item in the **qas** list has the following keys: **answers**, **id**, **question**, and **is_impossible**.



In [19]:
cuad_data[0]['paragraphs'][0].keys()

dict_keys(['qas', 'context'])

In [7]:
# inspect the first contract's paragraph
cuad_data[0]["paragraphs"][0].keys() # each paragraph contains 'qas' and 'context'

# inspect the qas first; inspect the first item in the list
cuad_data[0]["paragraphs"][0]["qas"][0] # each qas item contains 'answers' which is a list of its own, 'id', 'question', and 'is_impossible'

# focussing on a single paragraph
single_paragraph = cuad_data[0]["paragraphs"][0]
# focussing on the same anchor text
anchor_text = single_paragraph["context"]

# loop through all questions inside the single paragraph
for qa in single_paragraph["qas"]:
  # skip categories where no terms were found
  if qa["is_impossible"]:
    continue

  clause_id = qa["id"].split("__")[-1]
  question = qa["question"]
  answers = [ans["text"] for ans in qa["answers"]]

  # print out the findings
  print(f"ID: {clause_id}")
  print(f"QUESTION: {question}")
  print(f"ANSWER: {answers}")
  print("="*40)

ID: Document Name
QUESTION: Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract
ANSWER: ['DISTRIBUTOR AGREEMENT']
ID: Parties
QUESTION: Highlight the parts (if any) of this contract related to "Parties" that should be reviewed by a lawyer. Details: The two or more parties who signed the contract
ANSWER: ['Distributor', 'Electric City Corp.', 'Electric City of Illinois L.L.C.', 'Company', 'Electric City of Illinois LLC']
ID: Agreement Date
QUESTION: Highlight the parts (if any) of this contract related to "Agreement Date" that should be reviewed by a lawyer. Details: The date of the contract
ANSWER: ['7th day of September, 1999.']
ID: Effective Date
QUESTION: Highlight the parts (if any) of this contract related to "Effective Date" that should be reviewed by a lawyer. Details: The date when the contract is effective 
ANSWER: ['The term of this  Agreement  shall be ten (10)                        

In [11]:
anchor_text

'EXHIBIT 10.6\n\n                              DISTRIBUTOR AGREEMENT\n\n         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.\n\n                                    RECITALS\n\n         A. The  Company\'s  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an "Energy  Saver"  which may be improved  or  otherwise  changed  from its present composition (the "Products").  The Company may engage in the business of selling other  products  or  other  devices  other  than  the  Products,  which  will be considered  Products if Distributor  exercises its options pursuant to Section 7 hereof.\n\n         B. Representations.  As an inducement to the Company to enter into this Agreement,  the  Distributor  has  represented  that  it has or  will  hav

In [8]:
# extract the unique clauses from the contracts
unique_clauses = set()

for contract in cuad_data:
  # access the paragraphs in each contract
  for paragraph in contract["paragraphs"]:
    # access the qas of each paragraph
    qas = paragraph["qas"]
    # extract the ids from each qa
    for qa in qas:
      if qa["is_impossible"]:
        continue
      clause_id = qa["id"].split("__")[-1]
      unique_clauses.add(clause_id)

print("Unique clauses found in the first 5 contracts:")
print(unique_clauses)
print(f"Number of unique clauses found: {len(unique_clauses)}")

Unique clauses found in the first 5 contracts:
{'Parties', 'Agreement Date', 'Affiliate License-Licensor', 'Covenant Not To Sue', 'Non-Compete', 'Governing Law', 'Rofr/Rofo/Rofn', 'Ip Ownership Assignment', 'Notice Period To Terminate Renewal', 'Audit Rights', 'Renewal Term', 'Non-Transferable License', 'No-Solicit Of Employees', 'Minimum Commitment', 'Anti-Assignment', 'Warranty Duration', 'Liquidated Damages', 'Source Code Escrow', 'Non-Disparagement', 'Change Of Control', 'Price Restrictions', 'No-Solicit Of Customers', 'Post-Termination Services', 'Unlimited/All-You-Can-Eat-License', 'Exclusivity', 'Cap On Liability', 'Irrevocable Or Perpetual License', 'Insurance', 'Most Favored Nation', 'Affiliate License-Licensee', 'Competitive Restriction Exception', 'Expiration Date', 'Uncapped Liability', 'Third Party Beneficiary', 'Volume Restriction', 'Effective Date', 'Revenue/Profit Sharing', 'Document Name', 'Termination For Convenience', 'License Grant', 'Joint Ip Ownership'}
Number of 

In [9]:
# HIGH PRIORITY CLAUSES THAT ARE SUBJECT TO INTENSE PRAGMATIC INFERENCE
# AND LITIGATION IN COURTS
tier_1_clauses = {
    'Audit Rights',
    'Termination For Convenience',
    'Most Favored Nation',
    'Non-Compete',
    'Insurance',
    'Minimum Commitment',
    'Post-Termination Services',
    'Warranty Duration'
}

# STRICT PROHIBITIONS BUT CAN ALSO BE LITIGATED IN COURTS BASED ON THE
# CONTEXT OF THE CONTRACT AND CASE
tier_2_clauses = {
    'Exclusivity',
    'Anti-Assignment',
    'Cap On Liability',
    'Competitive Restriction Exception',
    'Covenant Not To Sue',
    'No-Solicit Of Customers',
    'No-Solicit Of Employees',
    'Non-Disparagement',
    'Non-Transferable License',
    'Revenue/Profit Sharing',
    'Rofr/Rofo/Rofn',
    'Source Code Escrow',
    'Third Party Beneficiary',
    'Price Restrictions',
    'License Grant',
    'Affiliate License-Licensor',
    'Affiliate License-Licensee',
    'Ip Ownership Assignment',
    'Joint Ip Ownership',
    'Irrevocable Or Perpetual License',
    'Unlimited/All-You-Can-Eat-License',
    'Volume Restriction'
}

# DETERMINISTIC OPERATIONS AND CLAUSES; DATES, AMOUNT, NUMERICAL VALUES, AND
# OTHER CLEAR DETERMINISTIC OPERATIONS
tier_3_clauses = {
    'Agreement Date',
    'Document Name',
    'Effective Date',
    'Expiration Date',
    'Governing Law',
    'Liquidated Damages',
    'Notice Period To Terminate Renewal',
    'Renewal Term',
    'Parties',
    'Uncapped Liability'
}

In [67]:
vagueness_seed_set = {
    # ── Effort & diligence ──────────────────────────────────────────────
    "reasonable efforts", "best efforts", "commercially reasonable",
    "due diligence", "reasonable care", "good faith", "workmanlike manner",
    "best practices", "reasonable endeavours", "all reasonable steps",
    "every reasonable effort", "diligent efforts", "reasonable commercial efforts",
    "utmost care", "exercise of judgment", "commercially practicable",
    "economically reasonable", "technically feasible",
    "consistent with good industry practice", "as would a prudent operator",
    "acting reasonably", "using its discretion",

    # ── Time & urgency ──────────────────────────────────────────────────
    "promptly", "in a timely manner", "as soon as practicable",
    "without undue delay", "for a reasonable period",
    "termination of this Agreement", "from time to time", "periodic",
    "duration", "seasonable", "business hours", "within a reasonable time",
    "with all due speed", "expeditiously", "at the earliest opportunity",
    "without unnecessary delay", "within a commercially reasonable period",
    "in due course", "forthwith", "in due time", "on a timely basis",
    "in the near term", "shortly after", "when practicable",
    "upon reasonable notice", "reasonable notice period",

    # ── Scope, degree & quantity ────────────────────────────────────────
    "material", "substantial", "limited", "relevant", "related", "generally",
    "appropriate", "similar", "de minimis", "significant", "incidental",
    "including but not limited to", "inter alia", "and/or",
    "save as otherwise provided", "appreciable", "meaningful", "non-trivial",
    "measurable", "proportionate", "commensurate", "reasonably proportionate",
    "unduly burdensome", "reasonably necessary", "to the extent practicable",
    "to a reasonable extent", "without limitation", "as applicable",
    "where relevant", "as appropriate", "to the extent required",

    # ── Harm, change & threshold ────────────────────────────────────────
    "material adverse effect", "material breach", "material adverse change",
    "material adverse impact", "material adverse consequence",
    "materially and adversely", "substantial impairment", "material disruption",
    "material deviation", "materially prejudice", "disproportionate impact",
    "unreasonable hardship", "undue prejudice", "undue harm", "undue risk",

    # ── Necessity & discretion ──────────────────────────────────────────
    "necessary", "sole discretion", "need to know", "confidential nature",
    "adequate", "satisfactory", "proper", "intended purpose",
    "not to be unreasonably withheld", "mutual satisfaction", "at its option",
    "consultation", "absolute discretion", "unfettered discretion",
    "not to be unreasonably delayed", "not to be unreasonably conditioned",
    "without arbitrary restriction", "reasonably required",
    "reasonably requested", "if deemed appropriate", "as deemed necessary",
    "in its reasonable opinion", "acting in good faith",
    "in its reasonable judgment", "as it sees fit", "as directed",

    # ── Industry norms & quality ────────────────────────────────────────
    "customary", "ordinary course of business", "industry standard",
    "standard practice", "normally", "comparable", "acceptable",
    "conventional", "fit for purpose", "first-class condition",
    "commercially sensitive", "prevailing market practice",
    "generally accepted practice", "market standard",
    "accepted industry norms", "standard market terms",
    "customary market conditions", "in accordance with accepted methods",
    "consistent with past practice", "as is customary",
    "in accordance with best available techniques",
    "reasonable engineering standards", "professionally acceptable",
    "to a professional standard", "of merchantable quality",
    "of satisfactory quality",

    # ── Knowledge, intent & foresight ──────────────────────────────────
    "foreseeable", "contemplated", "intended", "anticipated", "applicable",
    "knowledge", "directly or indirectly", "disclosed in confidence",
    "all copies", "survive", "mutual agreement", "substantially similar",
    "reasonable expectations", "actual knowledge", "constructive knowledge",
    "reasonably should have known", "to the best of its knowledge",
    "as far as it is aware", "reasonably foreseeable",
    "unforeseen circumstances", "unanticipated events",
    "beyond reasonable expectation", "reasonable belief", "bona fide belief",
    "reasonable grounds", "having regard to all circumstances",

     # ── Confidentiality & information ───────────────────────────────────
    "proprietary information", "non-public information",
    "sensitive business information", "trade secrets",
    "sufficiently confidential", "reasonably considered confidential",
    "maintained in confidence", "treated as confidential",
    "in accordance with confidentiality obligations",

    # ── Financial & commercial terms ────────────────────────────────────
    "commercially attractive", "economically viable",
    "commercially justifiable", "at a reasonable price", "fair market value",
    "arm's length", "at prevailing rates", "on reasonable commercial terms",
    "on competitive terms", "at a rate reflecting market conditions",
    "at cost", "without unreasonable mark-up", "reasonable compensation",
    "reasonable fees",

    # ── Survival, agreement & modification ─────────────────────────────
    "notwithstanding the foregoing", "without prejudice to",
    "subject to the foregoing", "except as otherwise agreed",
    "unless otherwise specified", "where not inconsistent", "insofar as",
    "to the fullest extent permitted by law", "as may be amended",
    "as modified from time to time", "by mutual written consent",
}

len(vagueness_seed_set)

206

### Extracting clause categories and texts

Using the dataset available, the following needs to be extracted:

*   Document ID (for cross referencing later if needed)
*   Clause category/ID
*   Text from the clause that describes the contract
*   The tier it belongs to
*   The set of vague terms contained in it

Furthermore, we also need to check whether the clause is:

*   Lethal - belongs to tier 1
*   Has context risk - belongs to tier 2

All of these will be stored as a dictionary, and housed in a list



In [68]:
clause_and_terms = list()

for contract in cuad_data:
  # access the paragraphs in each contract
  for paragraph in contract["paragraphs"]:
    # access the qas of each paragraph
    qas = paragraph["qas"]
    # extract the ids from each qa
    for qa in qas:
      # check_lethality flags if the clause belongs to tier 1
      # context_risk flags if the clause belongs to tier 2
      check_lethality, context_risk = False, False
      tier = 0

      if qa["is_impossible"]:
        continue
      clause_id = qa["id"].split("__")[-1]

      # check if the clause id belongs to tier 3, if yes, discard it
      if clause_id in tier_3_clauses:
        continue

      # check tier and assign label
      if clause_id in tier_1_clauses:
        tier = 1
      elif clause_id in tier_2_clauses:
        tier = 2

      clause_text = qa["answers"][0]["text"]
      contract_id = contract['title']
      vague_terms = {term for term in vagueness_seed_set if term in clause_text}


      # check the lethality of the clause
      if tier == 1 and vague_terms:
        check_lethality = True
        context_risk = False
      elif tier == 2 and vague_terms:
        check_lethality = False
        context_risk = True

      if check_lethality or context_risk:
        clause_term_dict = {
            "contract_id": contract_id,
            "clause_category": clause_id,
            "clause_text": clause_text,
            "tier": tier,
            "vague_terms": vague_terms,
            "is_lethal": check_lethality,
            "has_context_risk": context_risk
        }

        clause_and_terms.append(clause_term_dict)

In [69]:
print(f"Extracted data: {len(clause_and_terms)}")

Extracted data: 1428


In [70]:
from prompt_toolkit.shortcuts import print_container
import random

sampling_size = int(len(clause_and_terms) * 0.20)
compressed_sampling_size = int(sampling_size * 0.05)

for i in range(compressed_sampling_size):
  random_idx = random.randint(0, len(clause_and_terms)-1)

  print(f"CLAUSE CATEGORY: {clause_and_terms[random_idx]["clause_category"]}")
  print(f"CLAUSE TEXT: {clause_and_terms[random_idx]["clause_text"]}")
  print(f"TIER: {clause_and_terms[random_idx]["tier"]}")
  print(f"VAGUE TERMS: {clause_and_terms[random_idx]["vague_terms"]}")
  print(f"IS LETHAL: {clause_and_terms[random_idx]["is_lethal"]}")
  print(f"HAS CONTEXT RISK: {clause_and_terms[random_idx]["has_context_risk"]}")
  print("="*40, end="\n\n")

CLAUSE CATEGORY: Exclusivity
CLAUSE TEXT: For the avoidance of doubt, the parties acknowledge that the foregoing restriction applies only to persistent sponsorship placement as judged by Sponsor at its discretion, and not to run-of-site banner advertisements or other rotating promotional placements.
TIER: 2
VAGUE TERMS: {'knowledge'}
IS LETHAL: False
HAS CONTEXT RISK: True

CLAUSE CATEGORY: Insurance
CLAUSE TEXT: During the Term of this Agreement and for a period of one (1) year after the termination or expiration of this Agreement, GSK shall obtain and/or maintain at its sole cost and expense, product liability insurance (including any self-insured arrangements) in amounts which are reasonable and customary in the U.S. pharmaceutical industry for companies of comparable size and activities
TIER: 1
VAGUE TERMS: {'and/or', 'comparable', 'customary'}
IS LETHAL: True
HAS CONTEXT RISK: False

CLAUSE CATEGORY: Non-Compete
CLAUSE TEXT: During the term of this Agreement, and for a period of t

Right now, each data point `clause_and_terms` houses the vague clauses, vague terms, and the accompanying signals such as lethality and context risks.

From this, we move on to semantic mapping with the case law from the Harvard Corpus, i.e., [COLD Cases](https://huggingface.co/datasets/harvard-lil/cold-cases).

The `clause_text` present in each data point will be used to semantically map the right case law from this corpus which will be the judicial prose of the triplet.



## Semantic Mapping with Harvard LIL COLD Cases

Taking `clause_and_terms`, we will use the `clause_text` key/item and encode it into its **vector embeddings**.

A judge will not always write the contractual clauses as and how it appears in a contract in his reasoning; hence, string matching will fail. We need to map them in a dense **vector space**.

In [71]:
from sentence_transformers import SentenceTransformer
import numpy as np

# use the latest embedding models available in huggingface
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# vectorize the clause_texts in each data point in clause_and_terms
anchor_clause_texts = [data_point["clause_text"] for data_point in clause_and_terms]
anchor_clause_embeddings = embedding_model.encode(
    anchor_clause_texts,
    show_progress_bar=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/45 [00:00<?, ?it/s]